<p style="font-family: Cambria; text-align: center; font-size: 48px;">I. DATA PRE-PROCESSING

In [27]:
import pandas as pd
import numpy as np
import os
import glob
import re

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>The pd.concat() approach was used because the dataset was distributed across multiple CSV files, with each file containing records for an individual patient. After loading and preprocessing each file separately including extracting and appending a standardized Patient ID ,the data existed as multiple independent DataFrames. For effective analysis, such as merging with demographic data, performing aggregations, and generating visualizations, it was necessary to consolidate all records into a single dataset. The pd.concat(all_data, ignore_index=True) function enables this by vertically combining all individual DataFrames into one unified structure. The use of ignore_index=True ensures that the final dataset maintains a clean, continuous index, eliminating inconsistencies or duplicate indices from the original files. This approach is essential for maintaining data integrity and enabling efficient downstream analysis within a single, cohesive DataFrame

In [28]:
#1. Get all CSV files
files = glob.glob(r"C:\Users\bgaya\OneDrive\Desktop\HUPA-UC Diabetes Dataset\*.csv")
#2. Create empty list
all_data = []

# 3. Loop through each file and read it
for file in files:
    df = pd.read_csv(file, sep=';')
    
    # Extract full ID like HUPA0001P(Extract Patient ID from filename)
    patient_id = re.search(r'HUPA\d+P', os.path.basename(file)).group()
    
    # Assign directly (no conversion)
    df['patient_id'] = patient_id
      all_data.append(df)

#Combine all files
df = pd.concat(all_data, ignore_index=True)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 309392 entries, 0 to 309391
Data columns (total 9 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   time                    309392 non-null  object 
 1   glucose                 309392 non-null  float64
 2   calories                309392 non-null  float64
 3   heart_rate              309392 non-null  float64
 4   steps                   309392 non-null  float64
 5   basal_rate              309392 non-null  float64
 6   bolus_volume_delivered  309392 non-null  float64
 7   carb_input              309392 non-null  float64
 8   patient_id              309392 non-null  object 
dtypes: float64(7), object(2)
memory usage: 21.2+ MB


 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>The time column is converted into a proper datetime data type to enable accurate and efficient time-based analysis. Raw time values are often stored as strings, which limits the ability to perform operations such as filtering, grouping, or extracting specific components. By using pd.to_datetime(), the data is transformed into a standardized datetime format that pandas can understand and manipulate.

In [30]:

df['time'] = pd.to_datetime(df['time'], errors='coerce')


 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>Once converted, extracting features like hour, date, and day becomes straightforward and highly valuable for analysis. The hour component helps analyze patterns within a day (e.g., glucose trends by hour), while the date allows aggregation at a daily level. Extracting the day (weekday name) enables identification of trends across days of the week, such as differences between weekdays and weekends. These derived features simplify time-series analysis, improve data grouping, and support meaningful visualizations and insights. Overall, this approach enhances the usability of the dataset and enables more detailed and structured analysis of temporal patterns.

In [31]:
df['hour'] = df['time'].dt.hour
df['date'] = df['time'].dt.date
df['date'] = pd.to_datetime(df['date']).dt.date
df['day'] = df['time'].dt.day_name()

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>The time column was dropped after extracting relevant components such as date, hour, and day because it became redundant for further analysis. Once the original timestamp was converted into a datetime format and its meaningful features were derived, retaining the full time column was no longer necessary. Keeping it could lead to unnecessary data duplication, increased memory usage, and potential confusion during analysis.

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>By removing the original time column, the dataset becomes more streamlined and easier to work with, focusing only on the specific temporal features required for analysis. This also improves readability and ensures that subsequent operations—such as grouping, aggregation, and visualization—are performed on clearly defined and relevant variables rather than raw timestamp data.

In [32]:
df = df.drop('time', axis=1)

Changed Data type for appropirate measurement of each column  with their ideal range  Steps as Integer for getting whole value 

In [33]:
df= df.astype({
    'glucose': 'float32',
    'calories': 'float32',
    'heart_rate': 'int16',
    'steps': 'int32',
    'basal_rate': 'float32',
    'bolus_volume_delivered': 'float32',
    'carb_input': 'float32'
})

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>The dataset was grouped by patient_id, date, and hour to create hourly-level summaries of patient activity and physiological measurements. This aggregation was performed to transform raw time-series data into a structured format suitable for analysis and modeling.

Continuous variables such as glucose, heart_rate, and basal_rate were summarized using the mean to represent the average physiological state within each hour.

Activity-related variables such as steps, calories, bolus_volume_delivered, and carb_input were summarized using the sum to capture total hourly accumulation.

After aggregation, data types were adjusted for accuracy and consistency:

heart_rate values were rounded and converted to Int16 since heart rate is a discrete measurement and should not contain fractional values.
steps were converted to Int32 to ensure efficient storage while supporting large accumulated values without overflow

In [34]:
hourly_daily_df = df.groupby(['patient_id', 'date', 'hour']).agg({
    'glucose': 'mean',
    'heart_rate': 'mean',
    'steps': 'sum',
    'calories': 'sum',
    'bolus_volume_delivered': 'sum',
    'basal_rate': 'mean',
    'carb_input': 'sum'
}).reset_index()

#Steps and Heart Rate are not in floating value
# Then convert after aggregation
hourly_daily_df['heart_rate'] = hourly_daily_df['heart_rate'].round().astype('Int16')
hourly_daily_df['steps'] = hourly_daily_df['steps'].astype('Int32')
hourly_daily_df

,patient_id,date,hour,glucose,heart_rate,steps,calories,bolus_volume_delivered,basal_rate,carb_input
0,HUPA0001P,2018-06-13,18,328.000000,84,54,25.196501,0.0,0.091667,0.0
1,HUPA0001P,2018-06-13,19,275.916656,96,1376,139.104004,0.0,0.075000,0.0
2,HUPA0001P,2018-06-13,20,141.750000,100,1884,173.557999,0.0,0.075000,0.0
3,HUPA0001P,2018-06-13,21,80.083336,96,160,94.828995,3.5,0.009722,4.0
4,HUPA0001P,2018-06-13,22,153.416672,91,213,71.886497,0.0,0.029167,0.0
...,...,...,...,...,...,...,...,...,...,...
25797,HUPA0028P,2022-05-18,8,125.250000,64,4,56.032616,0.0,0.000000,0.0
25798,HUPA0028P,2022-05-18,9,137.000000,70,209,71.951996,0.0,0.000000,0.0
25799,HUPA0028P,2022-05-18,10,161.166672,96,239,117.911339,4.0,0.000000,6.0
25800,HUPA0028P,2022-05-18,11,108.500000,95,15,85.083237,0.0,0.000000,0.0


In [35]:
print(hourly_daily_df.groupby('patient_id').size())

patient_id
HUPA0001P      342
HUPA0002P      266
HUPA0003P      315
HUPA0004P      266
HUPA0005P      322
HUPA0006P      192
HUPA0007P      322
HUPA0009P      319
HUPA0010P      249
HUPA0011P      321
HUPA0014P      320
HUPA0015P      317
HUPA0016P      320
HUPA0017P      300
HUPA0018P      325
HUPA0019P      310
HUPA0020P      239
HUPA0021P      196
HUPA0022P      336
HUPA0023P      327
HUPA0024P      243
HUPA0025P      335
HUPA0026P     3384
HUPA0027P    13776
HUPA0028P     2160
dtype: int64


# Data Quality check

In [36]:
# Check date range per patient
print(hourly_daily_df.groupby('patient_id')['date'].agg(['min', 'max', 'count']))

# Check for duplicates in P27
p27 = hourly_daily_df[hourly_daily_df['patient_id'] == 'P27']
print(p27.duplicated(['date', 'hour']).sum())

# Check unique dates
print(p27['date'].nunique())
print(p27['hour'].nunique())

                   min         max  count
patient_id                               
HUPA0001P   2018-06-13  2018-06-27    342
HUPA0002P   2018-06-13  2018-06-24    266
HUPA0003P   2018-06-13  2018-06-26    315
HUPA0004P   2018-07-09  2018-07-20    266
HUPA0005P   2018-07-09  2018-07-22    322
HUPA0006P   2018-07-09  2018-07-17    192
HUPA0007P   2018-09-19  2018-10-02    322
HUPA0009P   2018-09-19  2018-10-02    319
HUPA0010P   2018-11-07  2018-11-17    249
HUPA0011P   2018-11-07  2018-11-20    321
HUPA0014P   2018-11-07  2018-11-20    320
HUPA0015P   2019-03-27  2019-04-09    317
HUPA0016P   2019-03-27  2019-04-09    320
HUPA0017P   2019-03-28  2019-04-09    300
HUPA0018P   2019-07-03  2019-07-16    325
HUPA0019P   2019-07-03  2019-07-16    310
HUPA0020P   2019-07-03  2019-07-13    239
HUPA0021P   2019-07-03  2019-07-11    196
HUPA0022P   2020-01-17  2020-01-30    336
HUPA0023P   2020-01-17  2020-01-30    327
HUPA0024P   2020-01-20  2020-01-30    243
HUPA0025P   2020-01-16  2020-01-30

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em> Id number 26,27,28 having many records,In website they clearly mentioned this test where only conducted for 14 dats reords.

The dataset was filtered to retain only the first 14 days of observations for each patient. This was achieved by grouping the data by patient_id and selecting records where the date falls within 14 days from each patient’s initial recorded date. This step ensures a consistent observation window across all patients, enabling fair comparison and reducing variability caused by differing follow-up durations.

In [76]:
validation = hourly_daily_df.groupby('Patient ID')['Date'].agg(['min', 'max', 'nunique'])
validation['day_span'] = (validation['max'] - validation['min']).dt.days + 1

print(validation)

                  min        max  nunique  day_span
Patient ID                                         
HUPA0001P  2018-06-13 2018-06-27       15        15
HUPA0002P  2018-06-13 2018-06-24       12        12
HUPA0003P  2018-06-13 2018-06-26       14        14
HUPA0004P  2018-07-09 2018-07-20       12        12
HUPA0005P  2018-07-09 2018-07-22       14        14
HUPA0006P  2018-07-09 2018-07-17        9         9
HUPA0007P  2018-09-19 2018-10-02       14        14
HUPA0009P  2018-09-19 2018-10-02       14        14
HUPA0010P  2018-11-07 2018-11-17       11        11
HUPA0011P  2018-11-07 2018-11-20       14        14
HUPA0014P  2018-11-07 2018-11-20       14        14
HUPA0015P  2019-03-27 2019-04-09       14        14
HUPA0016P  2019-03-27 2019-04-09       14        14
HUPA0017P  2019-03-28 2019-04-09       13        13
HUPA0018P  2019-07-03 2019-07-16       14        14
HUPA0019P  2019-07-03 2019-07-16       14        14
HUPA0020P  2019-07-03 2019-07-13       11        11
HUPA0021P  2

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>The dataset was standardized by restricting observations to a fixed 14-day period for each patient. This was achieved by grouping the data by patient_id and selecting only those records where the date falls within 14 days from each patient’s first recorded observation date.

This temporal filtering ensures that all patients have an equal observation window, improving comparability across subjects and reducing bias introduced by varying follow-up durations. The approach aligns the dataset with a consistent short-term study design, making it suitable for fair time-series analysis and downstream modeling tasks.

In [37]:
# Keep only first 14 days per patient
hourly_daily_df['date'] = pd.to_datetime(hourly_daily_df['date'])

hourly_daily_df = hourly_daily_df.groupby('patient_id').apply(
    lambda x: x[x['date'] <= x['date'].min() + pd.Timedelta(days=14)]
).reset_index(drop=True)

C:\Users\bgaya\AppData\Local\Temp\ipykernel_19984\3828448335.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hourly_daily_df = hourly_daily_df.groupby('patient_id').apply(


In [77]:

hourly_daily_df.groupby('Patient ID')['Date'].agg(['min', 'max', 'count'])

,min,max,count
Patient ID,,,
HUPA0001P,2018-06-13,2018-06-27,342
HUPA0002P,2018-06-13,2018-06-24,266
HUPA0003P,2018-06-13,2018-06-26,315
HUPA0004P,2018-07-09,2018-07-20,266
HUPA0005P,2018-07-09,2018-07-22,322
HUPA0006P,2018-07-09,2018-07-17,192
HUPA0007P,2018-09-19,2018-10-02,322
HUPA0009P,2018-09-19,2018-10-02,319
HUPA0010P,2018-11-07,2018-11-17,249


In [38]:
print(hourly_daily_df.groupby('patient_id').size())

patient_id
HUPA0001P    342
HUPA0002P    266
HUPA0003P    315
HUPA0004P    266
HUPA0005P    322
HUPA0006P    192
HUPA0007P    322
HUPA0009P    319
HUPA0010P    249
HUPA0011P    321
HUPA0014P    320
HUPA0015P    317
HUPA0016P    320
HUPA0017P    300
HUPA0018P    325
HUPA0019P    310
HUPA0020P    239
HUPA0021P    196
HUPA0022P    336
HUPA0023P    327
HUPA0024P    243
HUPA0025P    335
HUPA0026P    360
HUPA0027P    338
HUPA0028P    347
dtype: int64


In [39]:
hourly_daily_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7527 entries, 0 to 7526
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   patient_id              7527 non-null   object        
 1   date                    7527 non-null   datetime64[ns]
 2   hour                    7527 non-null   int32         
 3   glucose                 7527 non-null   float32       
 4   heart_rate              7527 non-null   Int16         
 5   steps                   7527 non-null   Int32         
 6   calories                7527 non-null   float32       
 7   bolus_volume_delivered  7527 non-null   float32       
 8   basal_rate              7527 non-null   float32       
 9   carb_input              7527 non-null   float32       
dtypes: Int16(1), Int32(1), datetime64[ns](1), float32(5), int32(1), object(1)
memory usage: 353.0+ KB


# Precision Handling

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em> The selected numerical columns (glucose, calories, basal_rate, bolus_volume_delivered, and carb_input) were formatted to maintain consistent numerical precision across the dataset. A global display setting was applied to show floating-point values with two decimal places for improved readability. Additionally, the values were explicitly rounded to two decimal places and converted to float32 to optimize memory usage while preserving sufficient precision for analysis.

In [40]:

cols = ['glucose', 'calories', 
        'basal_rate', 'bolus_volume_delivered', 'carb_input']


pd.set_option('display.float_format', '{:.2f}'.format)

hourly_daily_df[cols] = hourly_daily_df[cols].round(2)
hourly_daily_df[cols] = hourly_daily_df[cols].astype('float32').round(2)

In [41]:
hourly_daily_df.dtypes


patient_id                        object
date                      datetime64[ns]
hour                               int32
glucose                          float32
heart_rate                         Int16
steps                              Int32
calories                         float32
bolus_volume_delivered           float32
basal_rate                       float32
carb_input                       float32
dtype: object

Category value 

# Transformation

### Bolus Insulin Categorization

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>The continuous variable bolus_volume_delivered was transformed into categorical groups using predefined clinical thresholds. This binning approach was applied using pd.cut() to convert raw insulin dosage values into interpretable categories: No insulin, Low dose, Medium dose, and High dose.

The purpose of this transformation was to simplify interpretation of insulin usage patterns, reduce noise from small numerical variations, and enable clearer comparison across patient behaviors. A small negative lower bound (-0.01) was used to ensure that zero values were correctly included in the “No insulin” category. The resulting categorical distribution was then analyzed using value_counts() to understand the frequency of each insulin usage level across the dataset.

In [42]:
bins = [-0.01, 0, 1.0, 2.5, np.inf]
labels = ['No insulin', 'Low dose', 'Medium dose', 'High dose']

hourly_daily_df['bolus_category'] = pd.cut(
    hourly_daily_df['bolus_volume_delivered'],
    bins=bins,
    labels=labels
)
hourly_daily_df['bolus_category'].value_counts()

bolus_category
No insulin     6409
High dose       601
Medium dose     269
Low dose        244
Name: count, dtype: int64

In [43]:
hourly_daily_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7527 entries, 0 to 7526
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   patient_id              7527 non-null   object        
 1   date                    7527 non-null   datetime64[ns]
 2   hour                    7527 non-null   int32         
 3   glucose                 7527 non-null   float32       
 4   heart_rate              7527 non-null   Int16         
 5   steps                   7527 non-null   Int32         
 6   calories                7527 non-null   float32       
 7   bolus_volume_delivered  7527 non-null   float32       
 8   basal_rate              7527 non-null   float32       
 9   carb_input              7527 non-null   float32       
 10  bolus_category          7523 non-null   category      
dtypes: Int16(1), Int32(1), category(1), datetime64[ns](1), float32(5), int32(1), object(1)
memory usage: 360.5+ 

# Changing Column names

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>The column names were reformatted from Python-style snake_case to Title Case to improve readability and interpretability for reporting and visualization purposes. This transformation does not affect the underlying data but enhances clarity for end users by making variable names more human-readable.

In [44]:
hourly_daily_df = hourly_daily_df.rename(columns={
    'patient_id': 'Patient ID',
    'date': 'Date',
    'hour': 'Hour',
    'glucose': 'Glucose',
    'heart_rate': 'Heart Rate',
    'steps': 'Steps',
    'calories': 'Calories',
    'bolus_volume_delivered': 'Bolus Volume Delivered',
    'basal_rate': 'Basal Rate',
    'carb_input': 'Carb Input',
    'bolus_category': 'Bolus Category'
})

In [45]:
#making sure everthing is in Upper case
hourly_daily_df['Patient ID'] = hourly_daily_df['Patient ID'].str.upper()


In [46]:
hourly_daily_df['Patient ID'].value_counts()

Patient ID
HUPA0026P    360
HUPA0028P    347
HUPA0001P    342
HUPA0027P    338
HUPA0022P    336
HUPA0025P    335
HUPA0023P    327
HUPA0018P    325
HUPA0005P    322
HUPA0007P    322
HUPA0011P    321
HUPA0014P    320
HUPA0016P    320
HUPA0009P    319
HUPA0015P    317
HUPA0003P    315
HUPA0019P    310
HUPA0017P    300
HUPA0004P    266
HUPA0002P    266
HUPA0010P    249
HUPA0024P    243
HUPA0020P    239
HUPA0021P    196
HUPA0006P    192
Name: count, dtype: int64

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>Patient id 21,06  inclusion was validated based on data completeness criteria. Each patient was assessed for total number of unique observation days, and a minimum threshold of 14 days was applied to ensure consistent temporal coverage across subjects. Patients not meeting this threshold were excluded to reduce bias introduced by incomplete time-series records and to maintain comparability in downstream analysis.
identified that one patient had significantly fewer observations (~8 days vs 14 days). To maintain consistency and avoid bias in analysis, I excluded incomplete records and retained only patients with sufficient data coverage

# Coverting and Extrat Dataframe as Excel file

In [47]:
hourly_daily_df.to_excel(
    r"C:\Users\bgaya\OneDrive\Desktop\hourly_daily_df.xlsx",
    index=False,
    float_format="%.2f"
)

# Demographic file

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>The Excel and CSV versions of the dataset were compared using Python to verify data consistency. A row-wise and value-wise comparison was performed, and similarity percentage was calculated to confirm whether both files contain identical information. so dropped one of the file amoung 2

In [64]:

#Load both files

excel_df = pd.read_excel(r"C:\Users\bgaya\Python HacK-May-HUPADiabetic Dataset\HUPA-UC Diabetes Dataset\T1DM_patient_sleep_demographics_with_race.xlsx")

csv_df = pd.read_csv(r"C:\Users\bgaya\Python HacK-May-HUPADiabetic Dataset\HUPA-UC Diabetes Dataset\T1DM_patient_sleep_demographics_with_race.csv")

In [65]:
#ensure same structure
excel_df = excel_df.sort_index(axis=1).sort_values(by=excel_df.columns.tolist()).reset_index(drop=True)
csv_df = csv_df.sort_index(axis=1).sort_values(by=csv_df.columns.tolist()).reset_index(drop=True)

In [66]:
#Check exact match (100% identical or not)
excel_df.equals(csv_df)


True

In [67]:
matches = (excel_df == csv_df).all(axis=1).sum()
total = len(excel_df)

percentage = (matches / total) * 100
print(f"Similarity: {percentage:.2f}%")

Similarity: 100.00%


Calculate % similarity (row-level match) similary is 100% so we could delete on of the file

In [ ]:
# delete the CSV file from dataset
file_path = r"C:\Users\bgaya\Python HacK-May-HUPADiabetic Dataset\HUPA-UC Diabetes Dataset\T1DM_patient_sleep_demographics_with_race.csv"

if os.path.exists(file_path):
    os.remove(file_path)
    print("CSV file deleted successfully.")
else:
    print("File not found.")

# Reading Excel File as New demo_df Dataframe

In [50]:
demo_df = pd.read_excel(r"C:\Users\bgaya\Python HacK-May-HUPADiabetic Dataset\HUPA-UC Diabetes Dataset\T1DM_patient_sleep_demographics_with_race.xlsx")
demo_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 7 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Patient_ID                    25 non-null     object 
 1   Age                           25 non-null     int64  
 2   Gender                        25 non-null     object 
 3   Race                          25 non-null     object 
 4   Average Sleep Duration (hrs)  25 non-null     float64
 5   Sleep Quality (1-10)          25 non-null     float64
 6   % with Sleep Disturbances     25 non-null     int64  
dtypes: float64(2), int64(2), object(3)
memory usage: 1.5+ KB


 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>Changing the case Python-style snake_case to Title Case to improve readability and interpretability for reporting and visualization purposes.
This transformation does not affect the underlying data but enhances clarity for end users by making variable names more human-readable.

In [51]:
demo_df.rename(columns={'Patient_ID': 'Patient ID'}, inplace=True)

In [52]:

# making sure all ID 's in Upper cases
hourly_daily_df['Patient ID'] = hourly_daily_df['Patient ID'].astype(str).str.strip().str.upper()
demo_df['Patient ID'] = demo_df['Patient ID'].astype(str).str.strip().str.upper()

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>The hourly patient-level dataset (hourly_daily_df) was merged with the demographic and sleep dataset (demo_df) using a left join on the common key Patient ID. This merging step was performed to enrich the physiological time-series data with patient-level demographic and sleep-related attributes.

<p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>A left join was chosen to ensure that all records from the hourly dataset were retained, while matching demographic information was appended where available. This integration enables a more comprehensive dataset that combines short-term physiological measurements with long-term patient characteristics, supporting deeper exploratory analysis and improved feature richness for downstream modeling tasks.

In [53]:
merged_df = pd.merge(
    hourly_daily_df,
    demo_df,
    on='Patient ID',
    how='left'
)

In [54]:
# checking for null values

merged_df['Patient ID'].isna().sum()

np.int64(0)

In [55]:
#Check unmatched patients:

set(hourly_daily_df['Patient ID']) - set(demo_df['Patient ID'])

set()

In [57]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7527 entries, 0 to 7526
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Patient ID                    7527 non-null   object        
 1   Date                          7527 non-null   datetime64[ns]
 2   Hour                          7527 non-null   int32         
 3   Glucose                       7527 non-null   float32       
 4   Heart Rate                    7527 non-null   Int16         
 5   Steps                         7527 non-null   Int32         
 6   Calories                      7527 non-null   float32       
 7   Bolus Volume Delivered        7527 non-null   float32       
 8   Basal Rate                    7527 non-null   float32       
 9   Carb Input                    7527 non-null   float32       
 10  Bolus Category                7523 non-null   category      
 11  Age                           

<p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em> Rest Date To readable format

In [61]:
merged_df['Date'] = pd.to_datetime(merged_df['Date']).dt.date

Negative values were identified in the Bolus Volume Delivered column, which are physiologically invalid as insulin dosage cannot be negative. These occurrences were likely due to data entry or sensor recording errors. To ensure data integrity, negative values were corrected by clipping them to zero using a lower bound transformation, preserving the remaining valid observations.

In [80]:
merged_df['Bolus Volume Delivered'] = merged_df['Bolus Volume Delivered'].clip(lower=0)
merged_df['Bolus Volume Delivered']

0      0.00
1      0.00
2      0.00
3      3.50
4      0.00
       ... 
7522   0.00
7523   0.00
7524   0.00
7525   0.00
7526   0.00
Name: Bolus Volume Delivered, Length: 7527, dtype: float32

Missing values were identified in the Bolus Category column. Since these missing entries correspond to cases where the bolus insulin volume is zero or not recorded, they were interpreted as no insulin administration. To maintain dataset completeness and ensure consistency between the numeric and categorical representations of insulin delivery, missing values were imputed as “No insulin”.

In [81]:
merged_df['Bolus Category'].isna().sum()

np.int64(4)

In [83]:
merged_df.loc[merged_df['Bolus Volume Delivered'] == 0, 'Bolus Category'] = 'No insulin'

In [84]:
#finding missing values
merged_df['Bolus Category'].isna().sum()

np.int64(0)

In [85]:
#validation step
merged_df.groupby('Bolus Category')['Bolus Volume Delivered'].describe()

C:\Users\bgaya\AppData\Local\Temp\ipykernel_19984\2379287072.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  merged_df.groupby('Bolus Category')['Bolus Volume Delivered'].describe()


,count,mean,std,min,25%,50%,75%,max
Bolus Category,,,,,,,,
No insulin,6413.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
Low dose,244.00,0.79,0.25,0.18,0.60,1.00,1.00,1.00
Medium dose,269.00,1.84,0.42,1.10,1.50,1.90,2.20,2.50
High dose,601.00,6.43,3.32,2.60,3.80,5.40,8.40,20.80


 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em> Two extreme values were identified in the Glucose column exceeding 400 mg/dL, which is beyond the typical measurement range of continuous glucose monitoring (CGM) sensors. Such values are considered sensor artifacts or measurement errors rather than true physiological readings.

To maintain data quality and ensure realistic physiological boundaries, these extreme values were handled by setting them to missing (NaN). This approach prevents distortion in statistical analysis and modeling while preserving transparency of data quality issues. A follow-up flagging mechanism can be used to track these corrected observations for audit purposes.

In [ ]:
merged_df[merged_df['Glucose'] > 400]

In [87]:
merged_df.loc[merged_df['Glucose'] > 400, 'Glucose'] = np.nan

In [93]:
merged_df.loc[merged_df['Glucose'] > 400, ['Patient ID', 'Date', 'Glucose']]

,Patient ID,Date,Glucose


 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em>Continuous Glucose Monitoring (CGM) devices are designed to track glucose within a limited measurement range and typically have an upper operational cap of around 400 mg/dL. When glucose levels exceed this threshold, the sensor may become saturated, leading to truncated, capped, or unreliable readings rather than precise physiological measurements.

Therefore, values above 400 mg/dL in CGM datasets are commonly treated as sensor artifacts or measurement limitations rather than accurate clinical glucose values.

# Extract Final cleaned File as Cleaned data file

In [96]:
# Save output
merged_df.to_excel(' 12_PythonPioneers_cleaned_data.xlsx', index=False,float_format="%.2f")
print("\nSaved to  12_PythonPioneers_cleaned_data.xlsx")


Saved to  12_PythonPioneers_cleaned_data.xlsx


In [71]:
merged_df.groupby('Patient ID')['Glucose'].mean().describe()

count    25.00
mean    153.94
std      27.18
min     112.68
25%     136.16
50%     152.10
75%     173.02
max     200.78
Name: Glucose, dtype: float64

In [92]:
merged_df.head()

,Patient ID,Date,Hour,Glucose,Heart Rate,Steps,Calories,Bolus Volume Delivered,Basal Rate,Carb Input,Bolus Category,Age,Gender,Race,Average Sleep Duration (hrs),Sleep Quality (1-10),% with Sleep Disturbances
0,HUPA0001P,2018-06-13,18,328.00,84,54,25.20,0.00,0.09,0.00,No insulin,34,Male,Other,6.30,4.50,80
1,HUPA0001P,2018-06-13,19,275.92,96,1376,139.10,0.00,0.08,0.00,No insulin,34,Male,Other,6.30,4.50,80
2,HUPA0001P,2018-06-13,20,141.75,100,1884,173.56,0.00,0.08,0.00,No insulin,34,Male,Other,6.30,4.50,80
3,HUPA0001P,2018-06-13,21,80.08,96,160,94.83,3.50,0.01,4.00,High dose,34,Male,Other,6.30,4.50,80
4,HUPA0001P,2018-06-13,22,153.42,91,213,71.89,0.00,0.03,0.00,No insulin,34,Male,Other,6.30,4.50,80


 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em> Sleep-related variables (Average Sleep Duration, Sleep Quality, and Percentage of Sleep Disturbances) are patient-level static attributes repeated across hourly observations. These features were not aggregated during time-series processing to avoid artificial weighting caused by repeated hourly entries. Instead, they were treated as invariant enrollment-level characteristics and handled separately to preserve their true scale and prevent bias in downstream analysis.

In [94]:
merged_df.shape

(7527, 17)

 <p style="font-family: Cambria; font-size: 16px; color: green;"><strong><em> The final processed dataset contains 7,527 rows and 17 columns, representing hourly(one patient-hour observation) aggregated records across all patients with physiological, activity, and demographic features included for analysis.